In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

q_e=-1.6e-19
m_e=9.1e-31
q_i=-q_e
ne=20000

u0 = 4*np.pi*1e-7
epsi0 = 8.854187817e-12
c=1/np.sqrt(u0*epsi0)

L = 1.0
ng = 256
dx = L/ng
dt = 0.5*dx/c
t_tot = 200*L/c

In [24]:
x_e =np.linspace(0, L, ne, endpoint=False)
x_i=np.linspace(0,L,ne,endpoint=False)
E0=1e-5
Ex=np.zeros(ng)
Ey =E0* np.sin(2*np.pi/L * np.arange(ng) * dx)
B0z=1e-6
Bz  = np.full(ng, B0z) + (E0/c) * np.sin(2*np.pi/L * np.arange(ng) * dx)  # consistent Bz

Jx=np.zeros(ng)
Jy=np.zeros(ng)


In [25]:
Ex_his=[]
Ey_his=[]
vx_his=[]
vy_his=[]
Jx_his=[]
Jy_his=[]
Bz_his=[]


In [26]:
vx=np.zeros(ne)
vy=np.zeros(ne)

In [27]:
def getJx(vx,x_e):
    j_e= (x_e // dx).astype(int) % ng
    w_right_e = (x_e - j_e * dx) / dx
    w_left_e= 1.0 - w_right_e

    Jx=np.bincount(j_e,weights=w_left_e  * q_e * vx, minlength=ng)
    Jx+=np.bincount((j_e+1) % ng,  weights=w_right_e * q_e * vx,  minlength=ng)
    return Jx/dx

In [28]:
def getJy(vy,x_e):
    j_e= (x_e // dx).astype(int) % ng
    w_right_e = (x_e - j_e * dx) / dx
    w_left_e= 1.0 - w_right_e

    Jy=np.bincount(j_e,weights=w_left_e  * q_e * vy, minlength=ng)
    Jy+=np.bincount((j_e+1) % ng,  weights=w_right_e * q_e * vy,  minlength=ng)
    return Jy/dx

In [29]:
def getEx(J_curr,Ex):
    Ex-=J_curr/epsi0*(dt)
    return Ex

In [30]:
def getBz(Ey,Bz):
    for i in range(ng):
        Bz[i]-=((Ey[(i+1)%ng]-Ey[i])/dx)*(dt/2)
    return Bz

In [31]:
def getEy(Ey,Bz,Jy):
    for i in range(ng):
        Ey[i]+=((Bz[i]-Bz[(i-1)%ng])/dx)*c**2*(dt) - Jy[i]/epsi0*(dt)
    
    return Ey

In [32]:
def field_par(x_e_curr, F):
    j = (x_e_curr // dx).astype(int) % ng
    w_right = (x_e_curr - j*dx) / dx
    w_left  = 1.0 - w_right
    return w_left*F[j] + w_right*F[(j+1)%ng]

In [33]:
t=0
while t<t_tot:
    Ex_his.append(Ex.copy())
    vx_his.append(vx.copy())
    Ey_his.append(Ey.copy())
    Bz_his.append(Bz.copy())
    vy_his.append(vy.copy())
    Jx_his.append(Jx.copy())
    Jy_his.append(Jy.copy())
    Jx=getJx(vx,x_e) 
    Ex=getEx(Jx,Ex) 
    Bz=getBz(Ey,Bz) 
    Jy=getJy(vy,x_e) 
    Ey=getEy(Ey,Bz,Jy) 
    Bz = getBz(Ey, Bz)
    
    vx_minus = vx + (q_e/m_e)*field_par(x_e,Ex)*(dt/2)
    vy_minus = vy + (q_e/m_e)*field_par(x_e,Ey)*(dt/2)

    # Magnetic rotation (Boris)
    Bz_e   = field_par(x_e, Bz)
    t_rot  = (q_e/m_e)*Bz_e*(dt/2)
    s_rot  = 2*t_rot/(1 + t_rot**2)

    vx_prime = vx_minus + vy_minus*t_rot
    vy_prime = vy_minus - vx_minus*t_rot

    vx_plus  = vx_minus + vy_prime*s_rot
    vy_plus  = vy_minus - vx_prime*s_rot

    # Half electric acceleration
    vx = vx_plus + (q_e/m_e)*field_par(x_e,Ex)*(dt/2)
    vy = vy_plus + (q_e/m_e)*field_par(x_e,Ey)*(dt/2)
    x_e=(x_e+vx*dt)%L 
    t=t+dt

/tmp/ipykernel_101401/47988319.py:25: RuntimeWarning: overflow encountered in multiply
  vx_prime = vx_minus + vy_minus*t_rot
/tmp/ipykernel_101401/1596760933.py:2: RuntimeWarning: invalid value encountered in cast
  j_e= (x_e // dx).astype(int) % ng


KeyboardInterrupt: 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Physical parameters ──────────────────────────────────────────────────────
q_e   = -1.6e-19
m_e   =  9.1e-31
epsi0 =  8.854187817e-12
u0    =  4*np.pi*1e-7
c     =  1/np.sqrt(u0*epsi0)

n0    = 20000 / 1.0          # particles/m (1D number density)
B0z   = 1e-6                 # T

wp  = np.sqrt(n0*q_e**2/(epsi0*m_e))   # plasma frequency
wc  = abs(q_e)*B0z/m_e                  # electron cyclotron frequency
wR  = 0.5*wc + 0.5*np.sqrt(wc**2 + 4*wp**2)   # R cutoff
wL  = -0.5*wc + 0.5*np.sqrt(wc**2 + 4*wp**2)  # L cutoff
wUH = np.sqrt(wp**2 + wc**2)                    # upper hybrid

print(f"ωp  = {wp:.4e} rad/s")
print(f"ωc  = {wc:.4e} rad/s")
print(f"ωR  = {wR:.4e} rad/s")
print(f"ωL  = {wL:.4e} rad/s")
print(f"ωUH = {wUH:.4e} rad/s")

# ── Simulation dispersion from FFT2D of Ey_his ───────────────────────────────
Ey_arr = np.array(Ey_his)           # shape: (Nt, ng)
Nt, Ng = Ey_arr.shape

Ek = np.fft.fft2(Ey_arr)
Ek = np.fft.fftshift(Ek)
power = np.abs(Ek)**2

dt_val  = 0.5*dx/c
freqs   = np.fft.fftshift(np.fft.fftfreq(Nt, d=dt_val)) * 2*np.pi   # rad/s
kmodes  = np.fft.fftshift(np.fft.fftfreq(Ng, d=dx))     * 2*np.pi   # rad/m

# ── Theoretical X-wave dispersion: (ck)² = ω²(1 - ωp²(ω²-ωp²)/[ω²(ω²-ωUH²)]) ──
w_th = np.linspace(0, freqs.max()*1.05, 5000)
with np.errstate(divide='ignore', invalid='ignore'):
    # X-wave: n² = (ck/ω)² = 1 - ωp²(ω²-ωp²)/(ω²(ω²-ωUH²))
    num   = wp**2 * (w_th**2 - wp**2)
    denom = w_th**2 * (w_th**2 - wUH**2)
    n2    = 1.0 - num/denom
    k_th  = (w_th/c) * np.sqrt(np.where(n2 > 0, n2, np.nan))

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

# simulation power
ax.pcolormesh(kmodes/wp*c, freqs/wp, np.log10(power + 1e-30),
              cmap='inferno', shading='auto', vmin=-3)

# theoretical branches
ax.plot(k_th*c/wp, w_th/wp, 'w--', lw=1.4, label='X-wave theory')

# cutoff/resonance lines
for w, label, col in [(wR, r'$\omega_R$', 'cyan'),
                       (wL, r'$\omega_L$', 'lime'),
                       (wUH,r'$\omega_{UH}$','orange'),
                       (wp, r'$\omega_p$', 'yellow')]:
    ax.axhline(w/wp, color=col, lw=0.8, ls=':', label=label)

ax.set_xlabel(r'$ck/\omega_p$')
ax.set_ylabel(r'$\omega/\omega_p$')
ax.set_title('ω–k diagram: X-wave (Ey)')
ax.set_xlim(0, kmodes.max()*c/wp)
ax.set_ylim(0, freqs.max()/wp)
ax.legend(fontsize=8, loc='upper left')
plt.tight_layout()
plt.savefig('xwave_dispersion.png', dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

w_plot = np.linspace(0.01*wp, 3.5*wp, 5000)
with np.errstate(divide='ignore', invalid='ignore'):
    num   = wp**2 * (w_plot**2 - wp**2)
    denom = w_plot**2 * (w_plot**2 - wUH**2)
    n2    = 1.0 - num/denom
    vph2  = np.where(n2 > 1e-6, 1.0/n2, np.nan)   # vφ²/c² = 1/n²

ax.plot(w_plot/wp, vph2, 'royalblue', lw=2, label=r'$v_\phi^2/c^2$  (X-wave)')
ax.axhline(1.0, color='gray', lw=0.8, ls='--', label=r'$v_\phi = c$')

for w, label, col in [(wR, r'$\omega_R$','red'),
                       (wL, r'$\omega_L$','green'),
                       (wUH,r'$\omega_{UH}$','orange'),
                       (wp, r'$\omega_p$','purple')]:
    ax.axvline(w/wp, color=col, lw=0.9, ls=':', label=label)

# shade stop-bands (n² < 0 → vφ²/c² < 0, wave evanescent)
w_arr = w_plot/wp
n2_arr = 1.0 - wp**2*(w_plot**2 - wp**2)/(w_plot**2*(w_plot**2 - wUH**2))
ax.fill_between(w_arr, -2, 0, where=(n2_arr < 0),
                alpha=0.15, color='red', label='Stop band')

ax.set_xlabel(r'$\omega / \omega_p$')
ax.set_ylabel(r'$v_\phi^2 / c^2$')
ax.set_title(r'Phase velocity squared — X-wave dispersion')
ax.set_ylim(-2, 6)
ax.set_xlim(0, 3.5)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('vphi2_xwave.png', dpi=150)
plt.show()

In [ ]:
from matplotlib.animation import FFMpegWriter

ig_probe = ng // 4          # probe at x = L/4

Ex_probe = np.array([e[ig_probe] for e in Ex_his])
Ey_probe = np.array([e[ig_probe] for e in Ey_his])

fig, ax = plt.subplots(figsize=(5, 5))
ax.set_xlim(Ex_probe.min()*1.3, Ex_probe.max()*1.3)
ax.set_ylim(Ey_probe.min()*1.3, Ey_probe.max()*1.3)
ax.set_xlabel('Ex  (V/m)')
ax.set_ylabel('Ey  (V/m)')
ax.set_title('Elliptical polarisation — E-field Lissajous')
ax.set_aspect('equal')
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)

trail_len = 60
line,  = ax.plot([], [], 'royalblue', lw=1.2, alpha=0.6)
point, = ax.plot([], [], 'o', color='tomato', ms=6)

def init():
    line.set_data([], [])
    point.set_data([], [])
    return line, point

def update(frame):
    lo = max(0, frame - trail_len)
    line.set_data(Ex_probe[lo:frame], Ey_probe[lo:frame])
    point.set_data([Ex_probe[frame]], [Ey_probe[frame]])
    return line, point

Nt = len(Ex_his)
ani = animation.FuncAnimation(fig, update, frames=Nt,
                               init_func=init, blit=True, interval=20)

writer = FFMpegWriter(fps=30, metadata={'title': 'X-wave elliptical polarisation'},
                      extra_args=['-vcodec', 'libx264'])
ani.save("EllipticalPolarisation.mp4", writer=writer, dpi=150)
plt.show()
print("Saved EllipticalPolarisation.mp4")